# Bootstrapping and Confidence Intervals  

Objectives:

- Use **bootstrapping** to approximate sampling distributions  
- Construct and interpret **bootstrap confidence intervals (CIs)**  
- Apply bootstrapping to:
  - a **single mean**
  - a **single proportion**

- Practice using the **tidyverse** and **infer** packages in R


## 0. Setup

Run the following code chunk to load the necessary packages and data.

We’ll use:

- `tidyverse` for data wrangling and visualization  
- `infer` for tidy inference and bootstrapping  
- `palmerpenguins` for a real-world dataset on penguins in Antarctica


In [ ]:
# Install packages if needed (uncomment the following lines)
# install.packages("tidyverse")
# install.packages("infer")
# install.packages("palmerpenguins")

library(tidyverse)
library(infer)
library(palmerpenguins)

## 0.1 Data Information:

### 0.1.1 About palmerpenguins
The **palmerpenguins** package in R provides a dataset featuring measurements of three species of penguins found in the Palmer Archipelago, Antarctica. There are two datasets included in the package, **penguins** and **penguins_raw**.

### 0.1.2 Data Set:

#### Name: #### 
* `penguins`: Includes measurements for penguin species, on islands of Palmer Archipelago


#### Variables: ####

The variables included in the dataset are  
* `species`: a factor denoting penguin species (Adélie, Chinstrap and Gentoo)
* `island`: a factor denoting island in Palmer Archipelago, Antarctica (Biscoe, Dream or Torgersen)
* `bill_length_mm`: a number denoting bill length (millimeters)
* `bill_depth_mm`: a number denoting bill depth (millimeters)
* `flipper_length_mm`: an integer denoting flipper length (millimeters)
* `body_mass_g`: an integer denoting body mass (grams)
* `sex`: a factor denoting penguin sex (female, male)
* `year`: an integer denoting the study year (2007, 2008, or 2009).

##  0.2 Questions of interest

- Estimate **mean flipper length** among penguins of the Palmer Archipelago
- Estimate **proportion of Adelie** penguins among penguins of the Palmer Archipelago


## 0.3 Exploratory data analysis (EDA)


In [ ]:
glimpse(penguins)

The `tidymodels` package provides options for handling missing data. For now we will simply use the `na.omit` function to remove any observations with missing data.

In [ ]:
penguins <- na.omit(penguins)
glimpse(penguins)

### 0.3.1 Numerical Summaries

In [ ]:
# Summary statistics
penguins |>
  summarise(across(flipper_length_mm, list(mean = mean, sd = sd, min = min, max = max)))

In [ ]:
# Sample mean
obs_mean_flipper <- penguins |>
  summarize(mean_flipper = mean(flipper_length_mm)) |>
  pull(mean_flipper)

obs_mean_flipper

### 0.3.2 Graphical Summaries

In [ ]:
# Histogram
penguins |>
  ggplot(aes(x = flipper_length_mm)) +
  geom_histogram(binwidth = 5, fill = "blue", color = "white") +
  labs(x = "Flipper length (mm)", y = "Count",
       title = "Distribution of flipper length (mm)") +
  theme_minimal()


In [ ]:
# Boxplot
penguins |>
  ggplot(aes(x = species)) +
  geom_bar(fill = "blue") +
  labs(x = "Penguin Species", y = "Count",
       title = "Distribution of penguin species") +
  theme_minimal()


---


### 1. Resampling Methods

A resampling method involves repeatedly drawing samples from a dataset and refitting a model to obtain additional information about that model.  

Example: Suppose we want to know the variability associated the sample correlation.
- Draw different samples from the dataset
- Calculate sample correlation for each sample
- Examine how sample correlation varies from sample to sample

Example: Suppose we want to estimate the performance of statistical model in population.
- Draw sample(s) from the dataset
- Fit a linear regression to (each) sample
- Examine how the regression model fits data not included in the sample

Two common resampling methods are **cross-validation(CV)** and **bootstrap**.  

- **Cross-Validation**: can be used to estimate the test error associated with a statistical method to evaluate its performance, to select "tuning" parameters for a model, or to select a model’s level of flexibility  


- **Bootstrap**: a flexible and powerful statistical tool that can be used to quantify the uncertainty associated with a given estimator or statistical learning method. eg. estimate the standard error of a coefficient, or a confidence interval for that coefficient.   

## 1.1 Bootstrap

The *bootstrap* is a flexible and powerful statistical tool that can be used to **quantify the uncertainty** associated with a given estimator or statistical learning method.  
  
  
For example, in the context of a linear regression model, bootstrap methods could be used to estimate the standard error of a coefficient in the linear regression model, and/or a confidence interval for that coefficient. 

### 1.1.1 Bootstrap: General idea

We wonder what would happen (how would a statistic(estimator) behave) in repeated samples from the population (i.e. a sampling distribution).  
  
BUT: We typically can’t get more samples from a population.  

**Basic idea**: Simulate a new sample by resampling (with replacement) from the _original sample_ and computing the same statistic (e.g. a slope).  

This is known as a *bootstrap sample* and we collect the statistics for many such samples to form a ***bootstrap distribution***.

Power of the bootstrap lies in the fact that it can be easily applied to a wide range of statistical learning methods, including some for which a measure of variability is otherwise **difficult to obtain** and is **not automatically output** by statistical software.

### 1.1.2 Bootstrap: Terminology

- **Bootstrap "population"**: The original sample represents the "population" of interest for the purposes of bootstrap procedures.
- **Bootstrap sample**: Sample (with replacement) from the original sample (using same sample size).
- **Bootstrap statistic**: Compute the statistic of interest for the bootstrap sample
- **Bootstrap distribution**: Collect the bootstrap statistic for many (1,000’s) samples.

## 1.3 Bootstrapping a Mean

In this section, we’ll construct a **bootstrap distribution** and a **bootstrap confidence interval** for a **population mean**.

We’ll use the variable `flipper_length_mm` from `penguins`.

### 1.3.1 `rep_sample_n` approach

This may be accomplished in a couple of related ways. One option is to use the `rep_sample_n` function from the `infer` package to repeatedly resample from the `penguins` dataset. The `penguins` dataset, after removing observations with missing data, has 333 observations. So we will select samples of `size` 333. We are sampling with replacement, as required for bootstrap samples, and we will resample 2000 times.

In [ ]:
set.seed(1234)
# your code here
boot2000 <- penguins |>
    rep_sample_n(size = 333, replace = TRUE, reps = 2000)

head(boot2000)
tail(boot2000)

Now, for each bootstrap sample, we will calculate the sample mean.

In [ ]:
boot2000_means <- boot2000 |>
    group_by(replicate) |>
    summarise(mean = mean(flipper_length_mm))

head(boot2000_means)
tail(boot2000_means)

Finally, we can visualize the distribution of the bootstrap sample point estimates (`boot2000_means`) we just calculated by plotting a histogram.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 7)
# your code here
boot_est_dist <- ggplot(boot2000_means, aes(x = mean)) +
   geom_histogram(fill = "blue", colour = "white") +
   xlab("Average Flipper Length (mm)") +
   ggtitle("Bootstrap distribution") +
   theme_minimal()

boot_est_dist

This bootstrap distribution could then be used to produce bootstrap confidence intervals, in a manner similar to that discussed below.



Alternatively, we could make use of other functions that are part of the `infer` package to create a workflow that is more consistent with past practice.

### 1.3.2 `infer` package workflow

The `infer` package is an R package for statistical inference. It allows us to create a sequence of steps necessary to perform statistical inference in a “tidy” fashion. The `infer` package provides the necessary functions to perform statistical inference.

### `specify` variables

The specify() function is used to choose which variable(s) in a data frame are the variable(s) of interest and will be the focus of our statistical inference.

### `generate` replicates

The `generate` function is used to generate replicates. The `generate` function’s first argument is `reps`, which sets the number of replicates we would like to generate. The second argument determines how those replicates will be produced. Setting this to `type = "bootstrap"` indicates that we want to perform bootstrap resampling.

### `calculate` summary statistics
The `calculate` function will return the observed statistic specified with the `stat` argument. If provided the output of `generate`, the `calculate` function will calculate the supplied `stat` for each replicate. Some options for the `stat` argument are "mean", "median", "sd", "prop", "diff in means", "diff in medians", "diff in props", "Chisq", "F", "slope", "correlation", "t", "z", "ratio of props", "odds ratio", "ratio of means".


### 1.4 Create a Bootstrap Distribution for the Mean

We will:

1. Use `specify()` to declare the response variable  
2. Use `generate(type = "bootstrap")` to resample with replacement  
3. Use `calculate(stat = "mean")` to compute the mean for each bootstrap sample  

We’ll create **2000 bootstrap resamples**.

In [ ]:
set.seed(123)  # for reproducibility

boot_flipper <- penguins |>
  specify(response = flipper_length_mm) |>
  generate(reps = 2000, type = "bootstrap") |>
  calculate(stat = "mean")

boot_flipper |> glimpse()

### 1.5 Visualize the Bootstrap Distribution

Plot a histogram of the bootstrap means.

In [ ]:
boot_flipper |>
  ggplot(aes(x = stat)) +
  geom_histogram(bins = 30, fill = "blue", color = "white") +
  labs(x = "Bootstrap means of flipper length (mm)",
       title = "Bootstrap distribution of mean flipper length") +
  theme_minimal()

### 1.6 Bootstrap Confidence Interval for the Mean

#### Confidence intervals

We’ll compute a **95% percentile confidence interval** for the mean flipper length. There are two common ways to compute such an interval.

### 1.6.1 Percentile method with `infer`

This method sets the lower endpoint of the confidence interval at the 2.5th percentile of the bootstrap distribution and similarly sets the upper endpoint at the 97.5th percentile. The resulting interval captures the middle 95% of the values of the sample mean in the bootstrap distribution.

We can compute the 95% confidence interval by giving the bootstrap distribution to the `get_confidence_interval()` function from the `infer` package, with the confidence level set to 0.95 and the confidence interval type to be "percentile".

In [ ]:
boot_ci_flipper <- boot_flipper |>
  get_confidence_interval(level = 0.95, type = "percentile")

boot_ci_flipper

Alternatively, we can visualize the interval by giving the bootstrap distribution to the `visualize()` function and adding a `shade_confidence_interval()` layer. We set the endpoints argument to be `boot_ci_flipper`.

In [ ]:
visualize(boot_flipper) + 
  shade_confidence_interval(endpoints = boot_ci_flipper)

### 1.6.2 Standard error method with `infer`

Recall the standard error method for constructing 95% confidence intervals for any distribution that is normally shaped, builds upon the fact that roughly 95% of the values lie within two standard deviations of the mean of the distribution. In the case of the bootstrap distribution, the standard deviation has a special name: the standard error.

So in our case, 95% of values of the bootstrap distribution will lie within  
$\pm 1.96 \text{ standard errors of }\bar{x}$. Thus, a 95% confidence interval is
$$\bar{x} \pm 1.96 SE_\bar{x} = (\bar{x} - 1.96 SE_\bar{x}, \bar{x} + 1.96 SE_\bar{x})$$


Computation of the 95% confidence interval can once again be done by giving the bootstrap distribution to the `get_confidence_interval()` function, but setting the `type` argument to "se". We must also specify the `point_estimate` in order to establish the centre of the confidence interval. This is `obs_mean_flipper`, the sample mean of the original ssample, that we calculated earlier.

In [ ]:
standard_error_flipper_ci <- boot_flipper |> 
  get_confidence_interval(level = 0.95, type = "se", point_estimate = obs_mean_flipper)

standard_error_flipper_ci

In [ ]:
visualize(boot_flipper) + 
  shade_confidence_interval(endpoints = standard_error_flipper_ci)

Note that the confidence intervals produced in each case are almost identical. This is to be expected since the bootstrap distribution is approximately normally distributed. If the bootstrap distribution is not normally distributed, constructing intervals using the standard error method has less validity.

Similarly, if the bootstrap distribution is not symmetric, one should use the percentile method with caution.

### 1.6.3 General construction of confidence intervals

Consider the general construction of confidence intervals for some unknown parameter $\theta$, based upon some estimator $\hat{\theta}$.

The general objective is to find $a$ and $b$ such that 

$$P(\hat{\theta} -  a \le \theta \le \hat{\theta} +b)  = 1- \alpha$$

for a $(1-\alpha)$100\% confidence interval.

How to find $a$ and $b$.

If 

$$P(\hat{\theta} -  a \le \theta \le \hat{\theta} +b)  = 1- \alpha$$

then

$$P( -a \le \theta - \hat{\theta} \le b)  = 1- \alpha$$

or 

$$P( -b \le \hat{\theta} - \theta \le a)  = 1- \alpha$$


One method of constructing a 95\% confidence interval is to approximate the margin(s) of error by considering the difference between the sample statistics and the target statistic. The margin(s) of error are a measures of how much the sample statistic is likely to deviate from the target value, either overestimating or underestimating. For bootstrap distributions that are symmetric, the margins of error represented by $a$ and $b$ will be equivalent, but this is not the case for skewed distributions. For a 95\% confidence interval, the margin(s) of error are associated with the lower 2.5th percentile and upper 2.5th percentile of your sample statistics-target statistic 

### 1.6.4 CI interpretation  

We are 95% confident that average flipper length among all penguins in the Palmer Archipelago is between 199.5 mm and 202.5 mm.

---
## 2. Bootstrapping a Proportion

Next, we’ll construct a bootstrap CI for a **population proportion**.

Let’s consider the proportion of penguins that belong to the **Adelie** species.

### 2.1 Create a binary indicator

We'll create a simplified binary variable `is_adelie` with levels `"Adelie"` and `"Not_Adelie"`.

In [ ]:
penguins_prop <- penguins |>
  drop_na(species) |>
  mutate(is_adelie = if_else(species == "Adelie", "Adelie", "Not_Adelie"))

penguins_prop |>
  count(is_adelie)

### 2.2 Compute the sample proportion

Compute the sample proportion of penguins that are Adelie.

In [ ]:
obs_prop_adelie <- penguins_prop |>
  specify(response = is_adelie, success = "Adelie") |>
  calculate(stat = "prop")

obs_prop_adelie

### 2.3 Bootstrap distribution for the proportion

Use bootstrapping to create a distribution of sample proportions.

In [ ]:
set.seed(234)

boot_prop_adelie <- penguins_prop |>
  specify(response = is_adelie, success = "Adelie") |>
  generate(reps = 2000, type = "bootstrap") |>
  calculate(stat = "prop")

boot_prop_adelie |>
  ggplot(aes(x = stat)) +
  geom_histogram(bins = 30, fill = "blue", color = "white") +
  labs(x = "Bootstrap proportions (Adelie)",
       title = "Bootstrap distribution of Adelie proportion") +
  theme_minimal()

### 2.4 95% CI for the proportion

Compute a 95% percentile CI for the proportion of Adelie penguins.

In [ ]:
boot_ci_prop_adelie <- boot_prop_adelie |>
  get_confidence_interval(level = 0.95, type = "percentile")

boot_ci_prop_adelie

### 2.4.1 CI interpretation  

We are 95% confident that between 0.384 and 0.493, or 38.4% and 49.3%, of all penguins in the Palmer Archipelago are Adelie penguins.